In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import csv
from functools import partial
import subprocess

## Load full Siege results

Load all Siege log files and collate.

In [ ]:
bash = lambda cmd: subprocess.check_output(cmd, shell=True).decode().splitlines()
find_logs    = lambda logdir : bash(f'for f in {logdir}/*.log; do echo $(basename -s ".log" $f); done')
find_benches = lambda logfile: bash(f'grep "^\./fl.*\.bin" {logfile} | xargs -L 1 -- basename -s ".bin"')
find_rets    = lambda logfile: bash(f'grep "^Returned" {logfile} | cut -f2 -d" " | tr -d " "')
find_cycles  = lambda logfile: bash(f'grep "^Total cycles" {logfile} | cut -f2 -d"=" | tr -d " "')

to_ints  = lambda xs : [int(x) for x in xs]
log_type = lambda logname : 'baseline' if logname == "seq" else 'par'
log_pes  = lambda logname : 1 if logname == "seq" else int(logname.split('_')[-1])

# Identify existing log files
logdir = './logs'
logfiles = find_logs(logdir)

# Parse each log
dfs = []
for logname in logfiles:
    l = f'{logdir}/{logname}.log'
    df = pd.DataFrame({
        'ret' : to_ints(find_rets(l)),
        'cycles' : to_ints(find_cycles(l)),
        'bench' : find_benches(l)
    })
    df['type'] = log_type(logname)
    df['cores']  = log_pes(logname)
    dfs.append(df)
siege_df = pd.concat(dfs)

# Filter by our interesting bechmarks
final_benches = ['Coins', 'Minimax', 'Mss', 'Queens', 'SumEuler', 'Tak', 'TreeSum']
siege_df = siege_df[siege_df['bench'].isin(final_benches)]

Now we insert derived columns.

In [ ]:
def calc_speedups(full_df):
    par_df = full_df[full_df['type']=='par']
    seq_df = full_df[full_df['type']=='baseline']
    par_df['time'] = par_df['cycles'] / 102e6
    
    def add_speedup(row):
        nonlocal seq_df
        seq_row = seq_df[seq_df['bench'] == row['bench']].iloc[0]
        if seq_row['ret'] != row['ret']:
            raise ValueError(f'Run did not return expected values: {row} vs {seq_row}')
        row['speedup'] = seq_row['cycles']/row['cycles']
        row['ideal']   = row['speedup']/row['cores']
        return row
        
    return par_df.apply(add_speedup, axis=1).sort_values(by=['cores'])
    
siege_df = calc_speedups(siege_df)

# Plot and export

Display graphs for speedup (vs sequential version), % of ideal speedup obtained. We also export CSV files for use in LaTeX later.

In [ ]:
display(px.line(siege_df, x='cores', y='speedup', color='bench'))
display(px.line(siege_df, x='cores', y='ideal', color='bench'))

In [ ]:
!mkdir csv_siege

def exportBenchCsvs(df, basename):
    benches = np.unique(df['bench'])
    for b in benches:
        df[df['bench']==b].to_csv(f'{basename}_{b}.csv', index=False, na_rep='nan')
        
exportBenchCsvs(siege_df,'csv_siege/summary')

In [ ]:
siege_df[siege_df['cores']==24]['ideal'].mean()
siege_df[siege_df['cores']==24][['bench','time']]